# 🚀 Inferencia de Alto Rendimiento (vLLM) para Qwen 3.8 y GLM 5.3

Notebook optimizado para modelos de última generación:
- 🧠 **huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF** (Inferencia ultra rápida con vLLM GGUF loader)
- ⚡ **SC117/Qwen3.8-Flash-Next-GSQ-RCO-abliterated-GGUF**
- 🛡️ **dealignai/GLM-5.3-CYBERSECURITY-FP8** (Requiere GPU con alta VRAM / A100)

Cuenta con túnel público gratuito **Cloudflare Tunnel** compatible con OpenAI y chat Gradio integrado.

## 1. Verificar GPU y VRAM Asignada
Asegúrate de tener un entorno GPU activado en **Entorno de ejecución > Cambiar tipo de entorno de ejecución** (T4, L4 o A100).

In [ ]:
#@title 🔍 Comprobar GPU
!nvidia-smi
import torch
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"\n✅ GPU Activa: {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("\n❌ No se detectó GPU. Activa GPU en Entorno de ejecución > Cambiar tipo de entorno.")

## 2. Configuración del Modelo
### 📊 Guía de compatibilidad de GPUs en Colab Pro:
- **T4 (16 GB)**: `Qwen/Qwen2.5-7B-Instruct`, `Qwen/Qwen2.5-Coder-7B-Instruct` o `Qwen/Qwen2.5-14B-Instruct-AWQ` (con contexto 4096).
- **L4 (24 GB)**: `Qwen/Qwen2.5-7B-Instruct`, `Qwen/Qwen2.5-14B-Instruct-AWQ`, `Qwen/Qwen2.5-Coder-14B-Instruct-AWQ`.
- **A100 (40 GB / 80 GB)**: `Qwen/Qwen2.5-14B-Instruct`, `Qwen/Qwen2.5-32B-Instruct-AWQ`, `Qwen/Qwen2.5-Coder-32B-Instruct-AWQ` o incluso `Qwen/Qwen2.5-72B-Instruct-AWQ` (en 80GB).

In [ ]:
#@title ⚙️ Parámetros del Modelo
MODEL_PRESET = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF (Q4_K)" #@param ["huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF (Q4_K)", "dealignai/GLM-5.3-CYBERSECURITY-FP8", "personalizado"]
CUSTOM_HF_ID = "" #@param {type:"string"}
MAX_MODEL_LEN = 8192 #@param [2048, 4096, 8192, 16384, 32768] {type:"raw"}
GPU_UTILIZATION = 0.90 #@param {type:"slider", min:0.70, max:0.95, step:0.05}
API_KEY = "colab-qwen-secret" #@param {type:"string"}
PORT = 8000

IS_GGUF = False
GGUF_REPO = ""
GGUF_FILE = ""
TOKENIZER_ID = ""

if MODEL_PRESET == "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF (Q4_K)":
    IS_GGUF = True
    GGUF_REPO = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF"
    GGUF_FILE = "Huihui-Qwen3.8-27B-abliterated-Q4_K.gguf"
    TOKENIZER_ID = "huihui-ai/Huihui-Qwen3.8-27B-abliterated"
    MODEL_ID = GGUF_FILE
elif MODEL_PRESET == "dealignai/GLM-5.3-CYBERSECURITY-FP8":
    IS_GGUF = False
    MODEL_ID = "dealignai/GLM-5.3-CYBERSECURITY-FP8"
else:
    MODEL_ID = CUSTOM_HF_ID.strip()
    IS_GGUF = MODEL_ID.endswith(".gguf")

print(f"🎯 Preset seleccionado: {MODEL_PRESET}")
if IS_GGUF:
    print(f"📦 Repositorio GGUF: {GGUF_REPO}")
    print(f"📄 Archivo: {GGUF_FILE}")
    print(f"🔤 Tokenizer base: {TOKENIZER_ID}")
else:
    print(f"🤖 Modelo HF: {MODEL_ID}")
print(f"📏 Contexto: {MAX_MODEL_LEN} tokens")
print(f"🔑 API Key: {API_KEY}")

## 3. Instalar Dependencias
Instalamos `vllm`, paquetes cliente (`openai`, `gradio`) y el binario de `cloudflared` para túnel seguro sin registrarse.

In [ ]:
#@title 📦 Instalar vLLM, Plugin GGUF y Cloudflared
print("⏳ Instalando vLLM, vllm-gguf-plugin y Cloudflared (toma ~1-2 min)...")
# Instalamos vllm y su plugin oficial para soporte de modelos GGUF
!pip install -q --no-warn-conflicts vllm vllm-gguf-plugin openai gradio huggingface_hub

# Descargar e instalar Cloudflared para el túnel público
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb
print("✅ vLLM y vllm-gguf-plugin instalados con éxito.")

## 4. Iniciar Servidor vLLM y Generar URL Pública

In [ ]:
#@title 🚀 Descarga y Arranque del Servidor vLLM
import subprocess
import time
import requests
import re
import os
from huggingface_hub import hf_hub_download

# Matar procesos previos si existen
!pkill -f vllm > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1
time.sleep(1)

# Descargar archivo GGUF específico si es un modelo GGUF
if IS_GGUF:
    print(f"📥 Descargando archivo GGUF '{GGUF_FILE}' desde '{GGUF_REPO}' (aprox. 16.8 GB)...\n")
    local_model_path = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE, local_dir="./model_dir")
    target_model = os.path.abspath(local_model_path)
    extra_flags = f"--tokenizer {TOKENIZER_ID} --trust-remote-code"
else:
    target_model = MODEL_ID
    extra_flags = "--trust-remote-code"
    if "FP8" in MODEL_ID.upper():
        extra_flags += " --quantization fp8"
    elif "AWQ" in MODEL_ID.upper():
        extra_flags += " --quantization awq"

vllm_cmd = (
    f"python3 -m vllm.entrypoints.openai.api_server "
    f"--model {target_model} "
    f"--port {PORT} "
    f"--gpu-memory-utilization {GPU_UTILIZATION} "
    f"--max-model-len {MAX_MODEL_LEN} "
    f"--api-key {API_KEY} "
    f"{extra_flags}"
)

print(f"⏳ Comando vLLM:\n{vllm_cmd}\n")
print("⏳ Iniciando servidor vLLM en segundo plano...\n")
with open("vllm.log", "w") as vllm_log:
    vllm_proc = subprocess.Popen(vllm_cmd, shell=True, stdout=vllm_log, stderr=vllm_log)

# Iniciar Cloudflare Tunnel
tunnel_cmd = f"cloudflared tunnel --url http://localhost:{PORT}"
with open("tunnel.log", "w") as tunnel_log:
    tunnel_proc = subprocess.Popen(tunnel_cmd, shell=True, stdout=tunnel_log, stderr=tunnel_log)

# Obtener URL pública de Cloudflare
public_url = None
for _ in range(30):
    time.sleep(2)
    try:
        with open("tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break
    except Exception:
        pass

if public_url:
    print("=" * 65)
    print("🎉 ¡TÚNEL PÚBLICO ACTIVADO CON ÉXITO!")
    print(f"🌐 Base URL: {public_url}/v1")
    print(f"🔑 API Key: {API_KEY}")
    print(f"🤖 Modelo cargado: {target_model}")
    print("=" * 65 + "\n")
else:
    print("⚠️ No se encontró la URL del túnel todavía. Revisa tunnel.log")

# Esperar disponibilidad de vLLM
print("⏳ Esperando que vLLM cargue el modelo en GPU:")
ready = False
for i in range(120):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)
    print(".", end="", flush=True)

if ready:
    print("\n\n🚀 ¡Servidor vLLM LISTO Y OPERATIVO!")
else:
    print("\n\n⚠️ El servidor aún está cargando o hubo un aviso. Últimas líneas de vllm.log:")
    !tail -n 40 vllm.log

## 5. Prueba de Inferencia (Python OpenAI Client)

In [ ]:
#@title 🧪 Prueba Rápida de Streaming
from openai import OpenAI

client = OpenAI(
    base_url=f"http://localhost:{PORT}/v1",
    api_key=API_KEY
)

prompt = "Explica brevemente qué es la inferencia con vLLM y por qué es rápida."

print(f"Pregunta: {prompt}\n")
stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "Eres un asistente experto en Inteligencia Artificial."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.7,
    max_tokens=512,
    stream=True
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

## 6. (Opcional) Web Chat UI Interactivo con Gradio
Lanza un chat visual interactivo para probar el modelo directamente en el notebook o con enlace público compartible.

In [ ]:
#@title 💬 Iniciar Chat Gradio
import gradio as gr
from openai import OpenAI

local_client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key=API_KEY)

def chat_stream(message, history):
    messages = [{"role": "system", "content": "Eres un asistente inteligente y útil basado en Qwen."}]
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})
    
    response = local_client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=0.7,
        stream=True
    )
    
    bot_message = ""
    for chunk in response:
        delta = chunk.choices[0].delta.content or ""
        bot_message += delta
        yield bot_message

demo = gr.ChatInterface(
    fn=chat_stream,
    title=f"🤖 {MODEL_ID}",
    description="Inferencia en Colab Pro con vLLM y túnel Cloudflare",
    examples=[
        "Escribe una función en Python para ordenar una lista de diccionarios por clave.",
        "Explica la diferencia entre Fine-Tuning y RAG."
    ]
)
demo.launch(share=True, inline=True)

## 7. Monitorización de Recursos y Logs

In [ ]:
#@title 📊 Ver estado de GPU y últimos logs de vLLM
!nvidia-smi
print("\n--- Últimas 20 líneas de vllm.log ---")
!tail -n 20 vllm.log